PROGETTO 9: TRADUZIONE AUTOMATICA CON MODELLI SEQ2SEQ

Traduttore simultaneo, non traduce parola per parola, ascolta, comprende il concetto e lo ricostruisce in un'altra lingua

- Training di una pipeline Seq2Seq (cliccico ponte tra due lingue)
- Metrica BLEU giudice severo del nostro modello
- Stress-test, su strutture sintattiche complesse e analisi dei limiti della traduzione neurale classica.

Cominciamo delle fondamenta: l'addestramento

Addestramento della pipeline Seq2Seq
Costruzione di un ponte neurale tra due domini linguistici.
Immagina un traduttore che legge una frase in italiano, chiude gli occhi e riassume il testo (il vettore Thought) poi riapre gli occhi e traduce in inglese. L'obbiettivo non è la traduzione parola per parola ma la fedeltà semantica.

L'obbiettivo primario è mappare una sequenza di input in una lingua sorgente verso una sequenza di output in  una lingua target. Questo processo non avviene parola per parola, ma attraverso la creazione di una rappresentazione latente chiamata 'Thought Vector' che cattura il significato semantico globale della frase.
Utilizzeremo un dataset bilingue di piccole dimensioni per permettere la convergenza rapida, focalizzandoci sulla gestione del Teacher Forcing e sulla sincronizzazione temporale tra gli stati nascosti dell'Encodeer e del Decoder durante la fase di ottimizzazione dei pesi.

Ma come si coordinano questi due attori (encoder e decoder)?

Architettura e Dinamiche di Training
Componenti essenziali per l'apprendimento delle sequenze.
- Encoder RNN: elaborazione della frase sorgente per produrre un vettore di stato finale che riassume l'intero contesto informativo. L'Encoder è la nostra spugna, assorbe tutta la frase corrente e la spreme fino a ridurla ad un singolo vettore di stato.
- Decoder RNN: generazione della sequenza target partendo dal vettore di contesto e utiizzando i token precedenti come guida. Il Decoder è lo scultore, parte da quel vettore ed inizia ad estrarre le parole  una ad una
- Teacher Forcing: tecnica che passa la parola reale del dataset come input al decoder per prevenire la deriva dell'errore durante il training. Per correggere subito gli errori in fase di addestramento, per non fargli perdere il filo del discorso
- Padding e Masking: uniformazione delle lunghezze delle frasi per l'elaborazione a batch senza influenzare il calcolo del gradiente sulle parti vuote.

Il linguaggio di base è caotico per una macchina, quindi dobbiamo darle delle coordinate.
Ogni frase deve avere un segnale di inzio <start> ed uno di fine <end>. Questo permette al decoder di imparare quando iniziare la generazione e quando considerare conclusa la traduzione.
Poichè la traduzione è un problema di classificazione multiclasse su un intero vocabolario per ogni passo temporale, utilizzeremo la Sparse Categorical Crossentropy per minimizzare la discrepanza tra le parole predette e quelle reali (Loss function). Dice al modello quando la sua parola predetta è uguale a quelle reale in un mare di migliaia di vocaboli possibili.
La creazione di dizionari separati per le due lingue è cruciale. La dimensione degli embddding deve essere calibrata per bilanciare la ricchezza semantica e il numero di parametri del modello.

Il Calcolo della Cross-Entropy Temorale
Ottimizzazione su sequenze.
Immagina di correggere un compito, non guardi solo l'ultima parola per vedere se è giusta ma valutate l'intera sequenza, allo stesso modo la perdita viene valutata per ogni istante temporale e poi mediata. Questo permette al graediente di fluire all'indietro insegnando al decoder come parlare sia all'encoder come ascoltare meglio.
La funzione di perdita viene calcolata sommando l'errore di classificazione per ogni token della sequenza generata. Il gradiente viene poi propagato a ritroso attravrso il tempo sia nel decoder che nell'encoder.
L'errore totale per una frasi di lunghezza T viene normalizzato per permettere al modello di apprendere con stabilità indipendentemente dalla dimensione dell'input.

Una volta addestrato il modello, come facciamo a dargli un voto?

Valutazione con Metrica BLEU
Misurare la qualità della traduzione oltre la semplice accuratezza.
Valutare la traduzione di base è soggettivo, se chiedi a 10 persone di tradurre la stessa frase avrai 10 varianti corrette, ecco perchè l'accuratezza classica non funziona. Entra così in gioco la metrica BLEU (Bilingual Evaluation Understudy).
Immaginala come un critico letterario molto pignolo che controlla quanto la traduzione del computer assomiglia a una o più traduzioni umane di riferimento. Non guarda quindi solo le singole parole, ma la fluidità delle sequenze.
La metrica BLEU  confrontando la sovrapposizione di n-grammi tra la traduzione prodotta e quella  umana.
Analizzeremo come questo punteggio pesi la precisoine delle parole singole e delle sequenze corte, introducendo una penalità per frasi eccessivamente brevi che potrebbero tentare di 'ingannare' la metrica aumentando artificialmente la precisione.

Anatomia del Punteggio BLEU
Indicatori di fedeltà e fluidità linguistica
Il punteggio BLEU analizza gli n-grammi, ovvero blocchi di parole consecutive.
Se il modello azzecca le singole parole (unigrammi) ha una buona precisione lessicale, se il modello azzecca blocchi di 4 parole (4 grammi) allora ha capito la struttura sintattica.
- N-gram Precision: calcolo della percentuale di sequenze di parole (da 1 a 4) generate che appaiono effettivamente nella frase di riferimento
- Brevity Penalty: fattore correttivo che riduce il punteggio se la traduzione automatica è sensibilmente più corta della reference umana. Se una frase lunga viene tradotta con una sola parola corretta, la precisione sarebbe del 100% ma la traduzione sarebbe pessima. questo viene punito da BLEU
- Geometric Mean: combinazione delle precisioni dei diversi n-grammi per ottenere un valore unico tra 0 e 1 (o 0 e 100)
- Corpus BLEU: valutazione aggregata su un intero set di test per garantire una stima statisticamente significativa delle performance del modello

Nonostante la sua utilità, questo giudice ha dei limiti evidenti.

Limiti e Interpretazione del BLEU
- Mancanza di semantica: il limite più grande di BLEU è che non capisce i sinonimi. Se il modello traduce 'veloce' invece di 'rapido', la metrica penalizzerà il risultato nonostante il significato sia identico (toglierà punti). Alla fine è una metrica di sovrapposizione testuale non di comprensione semantica.
- Interpretazione dei valori: un punteggio BLEU sopra 0.4 è generalmente considerato eccellente, mentra un valore sotto 0.2 indicano traduzioni spesso frammentate o grammaticalmente errate.
- Riferimenti Multipli: la metrica permette di confrontare la traduzione prodotta con più varianti umane contemporaneamente, migliorando la capacità di catturare la flessibilità del linguaggio naturale.

Formalizzazione matematica di questo punteggio

La Formula del BLEU Score
Il calcolo finale combina la precisione modificata degli n-grammi con una funzione esponenziale basata sulla lunghezza della traduzione prodotta rispetto alla reference.
Il parametro w rappresenta il peso dato ad ogni n-grammo, solitamente impostato a 0.25 per coprire equamente i livelli da unigramma a quadrigramma.
E' una formula che ha dominato il campo per 20 anni

Ora che abbiamo il modello ed il modo per valutarlo, mettiamo alla prova nel mondo reale.

Stress-test su Frasi Complesse
La vera prova non sono le frasi fatte del tipo "il gatto è sul tavolo" ma le subordinate, ed è qui che emerge il dramma del touch vector. Immagina di dover riassumere un intero romanzo in un unica frase di 10 parole, perdereste dei dettagli. Nei modelli Seq2Seq classici, il vettore di contesto è un imbuto, e se la frase sorgente è troppo ricca l'informazioni iniziali si andranno a perdere, prima che il decoder possa elabolarle

Quindi quali sono i tipi di errori che incontreremo più spesso?

Analisi degli Errori Comuni
Sfide tipiche della traduzione neuronale non-attentiva
- Vanish Context: perdita di informazioni all'inizio di frasi molto lunghe a casua della saturazione del vettore di stato dell'encoder
- Allucinazioni: generazione di parole grammaticalmente corrette ma completamente estranee al significato della frase sorgente. Frasi bellissime, grammaticalmente perfette, ma che non hanno nulla a che fare con l'originale. Come un traduttore che si inventa la storia perchè non ha capito la storia sorgente.
- Word Ordering: difficoltà nel riordinare i costituenti della frase quando le due lingue hanno strutture grammaticali differenti.
- Out-of-Vocabulary (OOV): gestione fallimentare di termini rari o nomi propri non presenti nel vocabolario di training.

Fortunatamente abbiamo sviluppato delle strategie per mitigare questi problemi.

Strategie di Miglioramento
- Inversione dell'Input: una tecnica classica per migliorare le RNN Seq2Seq consiste nel fornire la frase sorgente al contrario all'encoder, riducendo la distanza temporale tra le prime parole sorgente e le prime parole target.
- Beam Search: invece di scegliere sempre la parola più probabile (Greedy Search), la Beam Search esplora diversi percorsi di traduzione per trovare la sequenza globalmente più coerente.
- Sottosegmentazion e (BPE) ci aiuta a non arrenderci davanti alle parole nuove, spezzettandole in prefissi e suffissi che il modello conosce già.

Ma perchè, in fondo, le frasi lunghe sono così difficili?

Dinamica del Thought Vector
Il limite fisico della rappresentazione (limite alla densità dell'informazione)
Se il vostro vettore di contesto ha 256 dimensioni e la frase ha 30 parole, ogni parola ha a disposizione meno di 9 dimensioni per farsi ricordare (256/30=8.53).
E' come cercare di far stare un intero armadio in una 24ore. Più la frase di allunga, più l'informazione per token diminuisce e la fedeltà crolla drasticamente. Questo è il motivo per cui è nato il meccanismo di Attention
Il Thought Vector agisce come un imbuto informativo. Tutta la complessità di una frase sorgente deve essere compressa in un vettore di dimensione fissa, creando una saturazione semantica.
Al crescere della lunghezza della sequenza N, la quantità di informazione per token che il vettore può trasportare diminuisce, portando al degrado della fedeltà della traduzione.


In [1]:
import os  #per configurazione ambiente
# Impostazione del backend di Keras: utilizziamo PyTorch per l'elaborazione dei tensori.
# Questa configurazione deve avvenire prima dell'importazione di Keras stesso.
os.environ["KERAS_BACKEND"] = "tensorflow"

import numpy as np
import tensorflow as tf
import keras #costruzione rete neurale
from keras import layers
from datasets import load_dataset
from nltk.translate.bleu_score import corpus_bleu #per valutazione Bleu

print("Backend Keras:", keras.backend.backend())

# ==============================================================================
# CONFIGURAZIONE DEI PARAMETRI DEL MODELLO (IPERPARAMETRI)
# ==============================================================================

# Numero massimo di frasi da caricare dal dataset per accelerare il tempo di addestramento.
MAX_SAMPLES = 20_000 
# Numero di campioni processati contemporaneamente in una singola iterazione di calcolo.
BATCH_SIZE = 256 #il modello non elabora 20.000 frasi tutte insieme fa gruppi di 256
# Numero totale di volte in cui l'intero dataset passerà attraverso la rete neurale.
EPOCHS = 100
# Dimensione del vettore denso che rappresenta ogni singola parola (spazio semantico).
EMBEDDING_DIM = 256 #ogni parola rappresentata mediante un vettore di 256 numeri
# Dimensione dello stato interno della cella LSTM (il "vettore concettuale").
LATENT_DIM = 512  
# Dimensione del vocabolario: quante parole distinte vogliamo che il modello riconosca.
VOCAB_SIZE = 15_000
# Lunghezza massima delle frasi (in numero di parole). Frasi più corte verranno riempite con zeri (padding).
SEQUENCE_LENGTH = 20  #ogni frase è rappresentata al massimo con 20 token

# ==============================================================================
# 1. CARICAMENTO E PREPARAZIONE DEL DATASET
# ==============================================================================

print("Caricamento del dataset OPUS Books (Inglese-Italiano)...")
# Carichiamo una porzione del dataset per bilanciare precisione e velocità di esecuzione.
dataset = load_dataset("Helsinki-NLP/opus_books","en-it",split=f"train[:{MAX_SAMPLES}]")
#ogni riga del dataset contiene la frase in ingle se e la frase in italiano

print(dataset)
print(dataset[0])
#separo source (inglese) e target (italiano)
source_texts = [] 
target_texts = []

# Iteriamo sui dati grezzi per separare le lingue e preparare il formato di destinazione.
for item in dataset:
    # Testo sorgente: lingua inglese.
    source_texts.append(item['translation']['en'])
    
    # Testo destinazione: lingua italiana.
    # Aggiungiamo i token speciali [start] e [end] per guidare il processo di generazione del Decoder.
    # [start] indica al modello dove iniziare la traduzione, [end] segnala quando fermarsi.
    target_texts.append(f"[start] {item['translation']['it']} [end]")

# Suddivisione del dataset in set di addestramento (90%) e set di valutazione (10%).
val_size = int(len(source_texts) * 0.1)
train_eng = source_texts[:-val_size]
train_ita = target_texts[:-val_size]
test_eng = source_texts[-val_size:]
test_ita = target_texts[-val_size:]

print(f"Esempio Input (EN): {train_eng[0]}")
print(f"Esempio Target (IT): {train_ita[0]}")

# ==============================================================================
# 2. VETTORIZZAZIONE DEL TESTO (TRASFORMAZIONE IN NUMERI)
# ==============================================================================

# Questa funzione pulisce il testo rimuovendo caratteri non necessari ma preservando i token speciali.
def custom_standardization(input_string):
    # Conversione in minuscolo.
    lowercase = tf.strings.lower(input_string)
    # Rimozione della punteggiatura standard, mantenendo però le parentesi quadre dei nostri token.
    # Include il supporto ai caratteri accentati tipici della lingua italiana.
    return tf.strings.regex_replace(lowercase, "[^a-z0-9\s\[\]àèìòùé]", "")

# Configurazione del layer per l'Inglese: trasforma le frasi in sequenze di interi.
# trasforma le parole in numeri ID
source_vectorizer = layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
    standardize=custom_standardization
)

# Configurazione del layer per l'Italiano.
# trasforma le parole in numeri ID
target_vectorizer = layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    # La lunghezza è +1 rispetto alla sorgente perché includiamo i token [start] e [end].
    output_sequence_length=SEQUENCE_LENGTH + 1, 
    standardize=custom_standardization
)

# Analisi dei testi per costruire la mappatura parola -> numero (vocabolario).
# analizza il traint set e genera il voabolario (inglese e italiano)
source_vectorizer.adapt(train_eng)
target_vectorizer.adapt(train_ita)

# ==============================================================================
# 3. PIPELINE DI DATI E TEACHER FORCING
# ==============================================================================

def format_dataset(eng, ita):
    # Trasformiamo il testo in sequenze numeriche tramite i vettorizzatori creati sopra.
    eng_vec = source_vectorizer(eng)
    ita_vec = target_vectorizer(ita)
    
    # Implementazione del Teacher Forcing:
    # Il Decoder riceve come input la frase italiana troncata dell'ultima parola.
    # L'obiettivo (target) della previsione è la stessa frase italiana ma spostata in avanti di uno, 
    # ovvero troncata del primo token ([start]).
    # In questo modo, data la parola X, il modello impara a prevedere la parola X+1.
    return (
        {
            "encoder_inputs": eng_vec,
            "decoder_inputs": ita_vec[:, :-1], 
        },
        ita_vec[:, 1:] 
    )

def make_dataset(eng_texts, ita_texts):
    # Creazione di un oggetto Dataset di TensorFlow per gestire il caricamento efficiente dei dati.
    dataset = tf.data.Dataset.from_tensor_slices((eng_texts, ita_texts))
    # Suddivisione in blocchi (batch) per l'elaborazione parallela.
    dataset = dataset.batch(BATCH_SIZE)
    # Applicazione del formato Teacher Forcing definito precedentemente.
    dataset = dataset.map(format_dataset, num_parallel_calls=tf.data.AUTOTUNE)
    # Mescolamento dei dati, precaricamento in memoria e caching per ottimizzare le prestazioni.
    return dataset.shuffle(2048).prefetch(16).cache()

# Creazione delle pipeline definitive per training e validazione.
train_ds = make_dataset(train_eng, train_ita)
val_ds = make_dataset(test_eng, test_ita)

# ==============================================================================
# 4. ARCHITETTURA DEL MODELLO SEQUENCE-TO-SEQUENCE (SEQ2SEQ) . Inizia la rete neurale
# ==============================================================================

# --- COMPONENTE ENCODER ---
# L'Encoder analizza la frase inglese e ne estrae il "significato" compresso.
encoder_inputs = keras.Input(shape=(None,), dtype="int64", name="encoder_inputs")
# Il layer Embedding converte l'indice della parola in un vettore denso.
# 'mask_zero=True' istruisce il modello a ignorare i pad (zeri di riempimento).
x = layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM, mask_zero=True)(encoder_inputs)
# la matrice degli embedding ha 15.000 * 256 valori, ogni parola ha quindi 256 parametri
# L'LSTM elabora la sequenza e restituisce lo stato finale della cella (h e c).
# Questi stati rappresentano il "Thought Vector" (vettore di pensiero), il ponte tra le due lingue.
encoder_outputs, state_h, state_c = layers.LSTM(LATENT_DIM, return_state=True)(x)
# LSTM legge tutta la frase e aggiorna progressivamente la propria meomria, alla fine 
# restituisce state_h (rappresentazione corrente) e state_c (memoria interna) 
# entrambi con dimensione 512

encoder_states = [state_h, state_c] #viene poi passato al decoder

# --- COMPONENTE DECODER ---
# Il Decoder riceve gli stati dell'Encoder e cerca di ricostruire la frase in italiano.
decoder_inputs = keras.Input(shape=(None,), dtype="int64", name="decoder_inputs")
decoder_embedding = layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM, mask_zero=True)
x = decoder_embedding(decoder_inputs)
# il decoder ha quindi un embedding italiano indipendente da quello inglese.

# Definiamo l'LSTM del Decoder.
decoder_lstm = layers.LSTM(LATENT_DIM, return_sequences=True, return_state=True)
# LSTM legge tutta la frase e aggiorna progressivamente la propria meomria, alla fine 
# restituisce state_h (rappresentazione corrente) e state_c (memoria interna) 
# entrambi con dimensione 512

# Inizializziamo il Decoder con gli stati prodotti dall'Encoder (il contesto della frase inglese).
# dice: encoder, parti dalla rappresentazione della frase inglese che l'encoder ha costruito.
decoder_outputs, _, _ = decoder_lstm(x, initial_state=encoder_states)

# Layer densi finali che mappano l'output della LSTM sulla probabilità di ogni parola nel vocabolario.
decoder_dense = layers.Dense(VOCAB_SIZE, activation="softmax")
decoder_outputs = decoder_dense(decoder_outputs)

# --- ASSEMBLAGGIO DEL MODELLO ---
# Uniamo Encoder e Decoder in un unico modello di training.
model = keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)
# il modello ha quindi due input: la frase inglese e la frase italiana precedente.

# Definizione della logica di ottimizzazione e della funzione di perdita.
model.compile(
    optimizer="rmsprop", 
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Visualizzazione della struttura logica del modello.
print(model.summary())

# TRAINING
# Avvio del processo di apprendimento.
print("\n--- Inizio dell'addestramento del modello Seq2Seq ---")
history = model.fit(train_ds, epochs=EPOCHS, validation_data=val_ds)

# ==============================================================================
# 5. LOGICA DI INFERENZA (GENERAZIONE DELLE TRADUZIONI)
# ==============================================================================

# Recuperiamo il vocabolario per poter convertire i numeri in parole leggibili.
ita_vocab = target_vectorizer.get_vocabulary()
ita_index_lookup = dict(zip(range(len(ita_vocab)), ita_vocab))

# Per la traduzione in tempo reale (inferenza), separiamo l'Encoder dal resto.
# Questo permette di codificare la frase inglese una sola volta.
print("Ottimizzazione dei modelli per la traduzione...")
inf_encoder_model = keras.Model(encoder_inputs, encoder_states)

# Definiamo i modelli di input per gli stati del Decoder durante il loop di generazione.
inf_decoder_state_input_h = keras.Input(shape=(LATENT_DIM,))
inf_decoder_state_input_c = keras.Input(shape=(LATENT_DIM,))
inf_decoder_states_inputs = [inf_decoder_state_input_h, inf_decoder_state_input_c]

# Riutilizzo dei layer già addestrati (Embedding e LSTM) per la generazione iterativa.
inf_dec_embed = decoder_embedding(decoder_inputs)
inf_dec_out, inf_h, inf_c = decoder_lstm(inf_dec_embed, initial_state=inf_decoder_states_inputs)
inf_dec_probs = decoder_dense(inf_dec_out)

# Modello Decoder dedicato all'inferenza: riceve un token e lo stato precedente, restituisce il prossimo token.
inf_decoder_model = keras.Model(
    [decoder_inputs] + inf_decoder_states_inputs,
    [inf_dec_probs, inf_h, inf_c]
)

def decode_sequence(input_sentence):
    # Conversione della frase inglese in numeri.
    tokenized_input = source_vectorizer([input_sentence])
    
    # 1. Fase di encoding: otteniamo il contesto della frase.
    states_value = inf_encoder_model.predict(tokenized_input, verbose=0)

    # 2. Inizializzazione della sequenza di destinazione con il token di inizio [start].
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = ita_vocab.index("[start]")

    decoded_sentence = ""
    stop_condition = False
    
    # Loop di generazione: prevediamo una parola alla volta finché non troviamo [end] o raggiungiamo il limite.
    while not stop_condition:
        # Previsione della probabilità della parola successiva.
        output_tokens, h, c = inf_decoder_model.predict([target_seq] + states_value, verbose=0)

        # Gestione della compatibilità tra backends (Torch/Tensorflow) per l'estrazione dell'indice massimo.
        probs = output_tokens[0, -1, :]
        sampled_token_index = np.argmax(probs.cpu().detach().numpy() if hasattr(probs, "cpu") else probs)
        
        # Conversione dell'indice numerico nella parola corrispondente.
        sampled_token = ita_index_lookup[sampled_token_index]

        # Condizioni di arresto.
        if sampled_token == "[end]" or len(decoded_sentence.split()) > SEQUENCE_LENGTH:
            stop_condition = True
        else:
            decoded_sentence += " " + sampled_token

        # Aggiornamento dello stato e del token di input per l'iterazione successiva.
        target_seq[0, 0] = sampled_token_index
        states_value = [h, c]
        
    return decoded_sentence.strip()

# ==============================================================================
# 6. VALUTAZIONE DELLE PRESTAZIONI (METRICA BLEU E TEST PRATICI)
# ==============================================================================

print("\n--- Analisi delle prestazioni e metrica BLEU ---")

references = []
hypotheses = []
# Preleviamo un campione ridotto per il test per non rallentare l'output.
subset_test_eng = test_eng[:50] 
subset_test_ita = test_ita[:50]

print("Generazione delle traduzioni per l'analisi statistica...")
for eng, ita_raw in zip(subset_test_eng, subset_test_ita):
    # Pulizia del riferimento reale (rimozione dei token speciali).
    ref_clean = ita_raw.replace("[start]", "").replace("[end]", "").strip().split()
    references.append([ref_clean])
    
    # Generazione della traduzione da parte del modello.
    pred = decode_sequence(eng)
    hyp_clean = pred.split()
    hypotheses.append(hyp_clean)

# Il punteggio BLEU misura la somiglianza tra la traduzione del modello e quella umana (1.0 = perfetta).
bleu_score = corpus_bleu(references, hypotheses)
print(f"Punteggio BLEU medio: {bleu_score:.4f}")

# Stress test: verifichiamo come si comporta il modello con frasi di diversa natura.
print("\n--- Test su frasi personalizzate ---")
custom_sentences = [
    "She is reading a book.",           # Frase semplice e diretta
    "I do not have time today.",        # Gestione della negazione
    "The weather is beautiful but cold.", # Utilizzo di congiunzioni
    "Happiness is the key to success.", # Vocabolario astratto
    "I went to the market to buy some fruit and vegetables for dinner." # Frase di lunghezza superiore
]

for sent in custom_sentences:
    print(f"Inglese: {sent}")
    print(f"Italiano: {decode_sequence(sent)}")
    print("-" * 30)

<>:86: SyntaxWarning: invalid escape sequence '\s'
<>:86: SyntaxWarning: invalid escape sequence '\s'
C:\Users\barbara\AppData\Local\Temp\ipykernel_42600\461013951.py:86: SyntaxWarning: invalid escape sequence '\s'
  return tf.strings.regex_replace(lowercase, "[^a-z0-9\s\[\]àèìòùé]", "")
c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Backend Keras: tensorflow
Caricamento del dataset OPUS Books (Inglese-Italiano)...


Dataset({
    features: ['id', 'translation'],
    num_rows: 20000
})
{'id': '0', 'translation': {'en': 'Source: Project Gutenberg', 'it': 'Source: www.liberliber.it/Audiobook available here'}}
Esempio Input (EN): Source: Project Gutenberg
Esempio Target (IT): [start] Source: www.liberliber.it/Audiobook available here [end]


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 256) │  3,840,000 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 256) │  3,840,000 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 512),     │  1,574,912 │ embedding[0][0],  │
│                     │ (None, 512),      │            │ not_equal[0][0]   │
│                     │ (None, 512)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │  1,574,912 │ embedding_1[0][0… │
│                     │ 512), (None,      │            │ lstm[0][1],       │
│                     │ 512), (None,      │            │ lstm[0][2]        │
│                     │ 512)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None,      │  7,695,000 │ lstm_1[0][0]      │
│                     │ 15000)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

...

None

--- Inizio dell'addestramento del modello Seq2Seq ---
Epoch 1/100


c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


71/71 ━━━━━━━━━━━━━━━━━━━━ 52s 670ms/step - accuracy: 0.0483 - loss: 7.4875 - val_accuracy: 0.0648 - val_loss: 6.3824
Epoch 2/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 48s 678ms/step - accuracy: 0.0664 - loss: 6.6748 - val_accuracy: 0.0884 - val_loss: 6.2636
Epoch 3/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 50s 706ms/step - accuracy: 0.0682 - loss: 6.6163 - val_accuracy: 0.0924 - val_loss: 6.2299
Epoch 4/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 49s 690ms/step - accuracy: 0.0851 - loss: 6.5665 - val_accuracy: 0.1223 - val_loss: 6.1657
Epoch 5/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 48s 674ms/step - accuracy: 0.0964 - loss: 6.4947 - val_accuracy: 0.1227 - val_loss: 6.1179
Epoch 6/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 49s 686ms/step - accuracy: 0.0979 - loss: 6.4605 - val_accuracy: 0.1272 - val_loss: 6.1008
Epoch 7/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 49s 695ms/step - accuracy: 0.0995 - loss: 6.4343 - val_accuracy: 0.1304 - val_loss: 6.0730
Epoch 8/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 48s 677ms/step - accuracy: 0.1020 - loss: 6.3936 - val_accuracy

c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


Punteggio BLEU medio: 0.0000

--- Test su frasi personalizzate ---
Inglese: She is reading a book.
Italiano: [UNK] disse il signor rochester
------------------------------
Inglese: I do not have time today.
Italiano: non posso dire
------------------------------
Inglese: The weather is beautiful but cold.
Italiano: e che cosa è accaduto
------------------------------
Inglese: Happiness is the key to success.
Italiano: e che cosa si [UNK]
------------------------------
Inglese: I went to the market to buy some fruit and vegetables for dinner.
Italiano: mi alzai e mi [UNK] a letto e la [UNK]
------------------------------
